# Assignment 1 - False Alarms

### Importing the required libraries

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import wfdb

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.feature_selection import RFE, RFECV
from imblearn.over_sampling import SMOTE

In [19]:
sns.set_theme(context="talk") 
plt.rcParams["figure.figsize"] = (10, 6)
RANDOM_STATE = 56

### Data Loading and Exploration

In [20]:
# Data Loading and Exploration

waveform_path = r"raw_data\waveforms"
event_path = r"raw_data\event_labels.csv"
split_path = r"raw_data\benchmark_data_split.csv"

event_df = pd.read_csv(event_path)
split_df = pd.read_csv(split_path)

event_df.sample(5)

In [21]:
# Taking a look at 5 random samples for event labels
event_df.sample(5)

In [22]:
# Taking a look at 5 random samples for split
split_df.sample(5)

In [23]:
# Merging event_df and split_df based on the record and event
df = event_df.merge(split_df, on=['record', 'event'], how='inner')

df.columns.to_list()

In [24]:
# Checking the summary stats of the dataframe
df.describe(include='all')

- Each event is unique, but the number of records can be different

In [25]:
# Checkinig percentage of True and False cases within the event dataframe
proportion = event_df['decision'].value_counts(normalize=True)*100
print(np.round(proportion,2))

In [26]:
# Checking distribution of split in terms of percentage
print(df['split'].value_counts(normalize=True)*100)

5037 alarm events with 4 columns namely record, event, decision and split. Class imbalance detected and data split in around 80-10-10 for train,validation,test respectively.

### Metadata Cleaning and Preprocessing

In [27]:
# Checking for any null values in metadata columns
df.isnull().sum()

In [28]:
# Mapping False as 0 and True as 1
df['decision'] = df['decision'].map({False:0,True:1})

In [29]:
# Unique patient record
print(f"Unique patient record: {event_df['record'].nunique()}")

In [30]:
# Total alarm events
print(f"Total cases: {len(event_df)}")

### Feature Extraction from Waveform files

#### Checking and getting summary stats for a single waveform

In [31]:
sample_one_path = r'raw_data\waveforms\003c13\003c13_0115'
record = wfdb.rdrecord(str(sample_one_path))
wfdb.plot_wfdb(record=record)

In [32]:
signals = record.p_signal
sig_names = record.sig_name

print(f"Signals: {signals}")
print(f"Signal's shape: {signals.shape}")
print(f"Signal Names: {sig_names}")

In [33]:
# Feature extraction for a single waveform
features = {}

for i, sig_name in enumerate(sig_names):
    signal = signals[:, i]
    signal_clean = signal[~np.isnan(signal)]

    if len(signal_clean) == 0:
        features[f'{sig_name}_missing_pct'] = 100
        continue

    features[f'{sig_name}_mean'] = np.mean(signal_clean)
    features[f'{sig_name}_std'] = np.std(signal_clean)
    features[f'{sig_name}_min'] = np.min(signal_clean)
    features[f'{sig_name}_max'] = np.max(signal_clean)
    features[f'{sig_name}_range'] = np.ptp(signal_clean)
    features[f'{sig_name}_median'] = np.median(signal_clean)
    features[f'{sig_name}_missing_pct'] = (np.isnan(signal).sum() / len(signal)) * 100

pd.DataFrame([features])

### Getting Summary Stats for all waveforms

In [34]:
# Getting Summary Stats for all waveforms
def extract_waveform_features(record_name, event_name, waveform_dir='raw_data/waveforms'):
    features = {}
    record_path = Path(waveform_dir) / record_name / event_name

    try:
        record = wfdb.rdrecord(str(record_path))
    except Exception:
        return {}

    signals = record.p_signal
    sig_names = record.sig_name

    for i, sig_name in enumerate(sig_names):
        signal = signals[:, i]
        signal_clean = signal[~np.isnan(signal)]

        if len(signal_clean) == 0:
            features[f'{sig_name}_mean'] = np.nan
            features[f'{sig_name}_std'] = np.nan
            features[f'{sig_name}_min'] = np.nan
            features[f'{sig_name}_max'] = np.nan
            features[f'{sig_name}_range'] = np.nan
            features[f'{sig_name}_median'] = np.nan
            features[f'{sig_name}_missing_pct'] = 100.0
            continue

        features[f'{sig_name}_mean'] = np.mean(signal_clean)
        features[f'{sig_name}_std'] = np.std(signal_clean)
        features[f'{sig_name}_min'] = np.min(signal_clean)
        features[f'{sig_name}_max'] = np.max(signal_clean)
        features[f'{sig_name}_range'] = np.ptp(signal_clean)
        features[f'{sig_name}_median'] = np.median(signal_clean)
        features[f'{sig_name}_missing_pct'] = (np.isnan(signal).sum() / len(signal)) * 100

    return features

In [35]:
# Extract features for all events
feature_list = []
for _, row in event_df.iterrows():
    feats = extract_waveform_features(row['record'], row['event'])
    feats['record'] = row['record']
    feats['event'] = row['event']
    feature_list.append(feats)

features_df = pd.DataFrame(feature_list)
df_full = df.merge(features_df, on=['record', 'event'], how='left')
df_full.sample(5)

### Waveform Data Cleaning

In [36]:
# Getting % of null values in different columns
feature_cols = [c for c in df_full.columns if c not in ['record', 'event', 'decision', 'split']]
missing_pct = df_full[feature_cols].isnull().mean()*100
missing_pct

In [37]:
# Dropping the column if more than 80% of data is missing
drop_features = missing_pct[missing_pct > 80].index.tolist()
feature_cols = [c for c in feature_cols if c not in drop_features]
df_full = df_full.drop(columns=drop_features)

In [38]:
# Dropping columns having high correlation of over 90%
corr_matrix = df_full[feature_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [column for column in upper_tri.columns if any(upper_tri[column] > 0.9)]
feature_cols = [c for c in feature_cols if c not in high_corr]
df_full = df_full.drop(columns=high_corr)

In [39]:
X = df_full[feature_cols].values
y = df_full['decision'].values

# Split indices
train_idx = df_full['split'] == 'train'
val_idx   = df_full['split'] == 'val'
test_idx  = df_full['split'] == 'test'

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]
X_test, y_test   = X[test_idx], y[test_idx]

In [40]:
from sklearn.metrics import f1_score, fbeta_score, roc_auc_score, classification_report, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, RocCurveDisplay

In [41]:
strategies = {
    'mean':   SimpleImputer(strategy='mean'),
    'median': SimpleImputer(strategy='median'),
    'knn':    KNNImputer(n_neighbors=5)
}

results = []
for name, imputer in strategies.items():
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(imputer.fit_transform(X_train))
    X_va   = scaler.transform(imputer.transform(X_val))

    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_va)
    y_prob = model.predict_proba(X_va)[:, 1]

    results.append({
        'Imputer':    name,
        'F1_val': f1_score(y_val, y_pred),
        'F2_val': fbeta_score(y_val, y_pred, beta=2),
        'AUC_ROC_val': roc_auc_score(y_val, y_prob),
        'Precision_val': precision_score(y_val, y_pred),
        'Recall_val': recall_score(y_val, y_pred),
    })

imputation_df = pd.DataFrame(results)
print("\nValidation Metrics by Imputation Strategy (Logistic Regression):")
print(imputation_df.to_string(index=False))

best_imputer_name = imputation_df.sort_values('F1_val', ascending=False).iloc[0]['Imputer']
best_imputer      = strategies[best_imputer_name]
print(f"\nBest imputer (by F1): {best_imputer_name}")

#Apply best imputer and scale
scaler          = StandardScaler()
X_train_scaled  = scaler.fit_transform(best_imputer.fit_transform(X_train))
X_val_scaled    = scaler.transform(best_imputer.transform(X_val))
X_test_scaled   = scaler.transform(best_imputer.transform(X_test))

- KNN Imputer works the best for Logistic Regresion, in our model

Instead of just using any model, we first create a class balance and compare the results between Logistic Regression and Random Forest.

### Using SMOTE

In [42]:
# Class Imbalance Handling (SMOTE)
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print("Class distribution after SMOTE:", np.bincount(y_train_smote))

In [43]:
# Modeling
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, class_weight='balanced')}

model_results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    fitted_models[name] = model

    for split_name, X_s, y_s in [('val', X_val_scaled, y_val), ('test', X_test_scaled, y_test)]:
        y_pred = model.predict(X_s)
        y_prob = model.predict_proba(X_s)[:, 1]
        model_results.append({
            'Model':     name,
            'Split':     split_name,
            'F1':        np.round(f1_score(y_s, y_pred), 4),
            'F2':        np.round(fbeta_score(y_s, y_pred, beta=2), 4),
            'AUC_ROC':   np.round(roc_auc_score(y_s, y_prob), 4),
            'Precision': np.round(precision_score(y_s, y_pred), 4),
            'Recall':    np.round(recall_score(y_s, y_pred), 4),
        })

model_results_df = pd.DataFrame(model_results)
print("\nModel Comparison (with SMOTE):")
print(model_results_df.to_string(index=False))

best_model_name = (model_results_df[model_results_df['Split'] == 'val']
                   .sort_values('F1', ascending=False).iloc[0]['Model'])
best_model = fitted_models[best_model_name]
print(f"\nBest model: {best_model_name}")

We use F1-score as the primary metric because:
1. Class imbalance: ~80% of alarms are FALSE alarms, so accuracy is misleading.
2. F1 is the harmonic mean of Precision and Recall, penalising models that sacrifice one for the other.

Accuracy not used as primary metric because it inflated by class imbalance hence does not reflect clinical utility.

Precision alone would mean we miss many real alarms and Recall alone would mean we would get excessive false alarms

- Random Forest works better than Logistic Regression on both validation and test dataset.
- Likely due to RF's ability to capture non-linear relationships in the waveform features.

In [44]:
# Classification Report for Best Model
print(f"\nClassification Report ({best_model_name}, Test Set) :")
y_test_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_test_pred, target_names=['False Alarm', 'True Alarm']))

In [45]:
# Confusion Matrix for Best Model
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred,
    display_labels=['False Alarm', 'True Alarm'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.show()

In [46]:
# ROC Curves for Both Models
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test_scaled, y_test, ax=ax, name=name)
ax.plot([0, 1], [0, 1], 'k--', label='Random Chance')
ax.set_title('ROC Curves')
ax.legend()
plt.tight_layout()
plt.show()

Higher AUC-ROC indicates better separation between True and False Alarms regardless of the chosen decision threshold

RFECV selects the optimal feature subset using cross-validated F1 scoring, helping to reduce overfitting and identify the most clinically relevant signals

In [47]:
# Recursive Feature Elimination with Cross Validation (RFECV).
rfecv = RFECV(
    estimator=LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    step=1,
    cv=StratifiedKFold(5),
    scoring='f1'
)
rfecv.fit(X_train_smote, y_train_smote)

X_train_rfe = rfecv.transform(X_train_smote)
X_test_rfe  = rfecv.transform(X_test_scaled)

rfe_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
rfe_model.fit(X_train_rfe, y_train_smote)

y_rfe_pred = rfe_model.predict(X_test_rfe)
y_rfe_prob = rfe_model.predict_proba(X_test_rfe)[:, 1]

print(f"\nRFECV Logistic Regression — Test Set:")
print(f"F1: {f1_score(y_test, y_rfe_pred):.4f}")
print(f"F2: {fbeta_score(y_test, y_rfe_pred, beta=2):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_rfe_prob):.4f}")
print(f"Recall: {recall_score(y_test, y_rfe_pred):.4f}")

selected_features = np.array(feature_cols)[rfecv.support_]
print("Selected features by RFECV:", selected_features)

Model’s best predictors are II_max, V_std, V_min.

### Feature Importance

In [48]:
# Feature Importance
if hasattr(best_model, 'coef_'):
    importances = best_model.coef_[0]
elif hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_

feat_importance = pd.DataFrame({
    'feature':        feature_cols,
    'importance':     importances,
    'abs_importance': np.abs(importances)
}).sort_values('abs_importance', ascending=False)

# Aggregate by waveform
feat_importance['waveform'] = feat_importance['feature'].apply(lambda x: x.split('_')[0])
waveform_importance = (feat_importance.groupby('waveform')['abs_importance'].mean().sort_values(ascending=False))
print("\nWaveform-level importance:")
print(waveform_importance)

In [49]:
# Plot top 20 features
fig, ax = plt.subplots(figsize=(10, 7))
feat_importance.head(20).plot.barh(x='feature', y='abs_importance', ax=ax, legend=False)
ax.set_xlabel('Absolute Importance')
ax.set_title(f'Top 20 Feature Importances — {best_model_name}')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

- Lead V and II has the most importance
- Model primarily relies on ECG leads V, II and aVR to predict false vs true alarms

#### Hyperparamter Tuning

In [50]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':      [100, 300, 500, 700],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.3],
    'class_weight':      ['balanced', 'balanced_subsample']
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=param_dist,
    n_iter=30,               \
    scoring='f1',
    cv=StratifiedKFold(5),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train_smote, y_train_smote)

print("Best params:", rf_search.best_params_)
print("Best CV F1:", rf_search.best_score_)

# Evaluate tuned model
best_rf = rf_search.best_estimator_
y_tuned_pred = best_rf.predict(X_test_scaled)
y_tuned_prob = best_rf.predict_proba(X_test_scaled)[:, 1]

print(f"\nTuned RF — Test Set:")
print(f"F1: {f1_score(y_test, y_tuned_pred):.4f}")
print(f"F2: {fbeta_score(y_test, y_tuned_pred, beta=2):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_tuned_prob):.4f}")
print(f"Recall: {recall_score(y_test, y_tuned_pred):.4f}")

##### Summary

- Missing waveform values were best handled with KNN imputation becasue it preserves relationships between signals better than mean or median imputation. 

- The modeling strategy combined Logistic Regression for linear baseline and Random Forest for capturing non-linear interactions, using statistical features from each waveform such as mean, std, min, max. 

- Random Forest is the best model  performing better with Median imputation than without, showing that imputing missing waveform features preserves predictive information. 

- Feature importance and RFECV indicate that ventricular ECG leads (V, II) are the strongest predictors with PLETH and ABP providing secondary support.